In [7]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain.tools import tool

load_dotenv()

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
)

@tool
def calcular_precio(cantidad: float, precio_unitario: float) -> str:
    """Calcula el precio total de un pedido."""
    total = cantidad * precio_unitario
    return f"El precio total es: {total:.2f} EUR"

tools = [DuckDuckGoSearchRun(), calcular_precio]

checkpointer = InMemorySaver()

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="Eres un asistente financiero experto. Responde siempre en español.",
    checkpointer=checkpointer,
)

chain_prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente financiero experto. Responde siempre en español."),
    ("human", "Pregunta: {pregunta}\n\nContexto web: {contexto}"),
])

config = {"configurable": {"thread_id": "demo-memoria"}}

# Ejemplo con LCEL
def buscar_docs(pregunta):
    search = DuckDuckGoSearchRun()
    return search.run(pregunta)

chain_completa = (
    RunnablePassthrough.assign(contexto=lambda x: buscar_docs(x["pregunta"]))
    | chain_prompt
    | llm
    | StrOutputParser()
)

resultado = agent.invoke( {"messages": [{"role": "user", "content": "Me llamo Juan, encantado."}]}, config)

resultado = agent.invoke(
    {"messages": [{"role": "user", "content": "¿Cómo me llamo?"}]},
    config,
)
print(resultado["messages"][-1].content)

# Streaming con LCEL
print("\n--- Streaming con LCEL ---")
for chunk in chain_completa.stream({"pregunta": "Explica los mercados financieros"}):
    print(chunk, end="", flush=True)   

Te llamas Juan. ¿En qué más puedo ayudarte?

--- Streaming con LCEL ---
Los mercados financieros son espacios, tanto físicos como virtuales, donde se compran y venden activos financieros como acciones, bonos, divisas, materias primas, entre otros. Su función principal es facilitar la transferencia de recursos entre quienes tienen excedentes de capital (inversores) y quienes necesitan financiamiento (empresas, gobiernos).

Estos mercados permiten la valoración de los activos, la liquidez para los inversores y la asignación eficiente del capital en la economía. Además, están influenciados por factores económicos, políticos y sociales, así como por el comportamiento de los inversores, que puede generar movimientos en los precios, como se observa en la volatilidad del bitcoin o en la fortaleza del dólar.

Un aspecto importante en los mercados financieros es la diversificación de las carteras, que ayuda a reducir riesgos y mejorar la rentabilidad. También existen diferentes técnicas para an